# NEXUS Enhanced — GRPO Training Notebook (**Enhanced**)

**Same pipeline as `grpo_colab_v2.ipynb`, with optional multi-incident rotation and scoped metrics `run_id`.**

Use **`grpo_colab_v2.ipynb`** for the simplest single-incident (INC003) path. Use **this notebook** when you want a **defined incident pool** (enterprise + EA + lighter variety) without editing code between runs.

- **Rotation:** `round_robin` (default) or `random` per reward episode (`NEXUS_INCIDENT_ROTATION`).
- **Pool:** `NEXUS_INCIDENT_POOL` (comma-separated) or defaults to `INC003,INC008,INC001`. Set `NEXUS_MULTI_INCIDENT=false` to lock to `NEXUS_INCIDENT_ID` only.
- **Metrics:** `NEXUS_GRPO_RUN_ID` tags `/reset` episodes so `GET /learning-curve?run_id=...` matches this Colab run.

## How to run (please follow order)

1. **Runtime:** GPU (e.g. T4).
2. Run cells **top to bottom** at least once per session: installs → **configuration** → environment → model → **training** → **plots**.
3. Edit the **configuration cell** for `BASE_URL`, incident pool, `GRPO_RUN_ID`, `ONE_ROUND_TRAINING`, and optional `NEXUS_*` env vars.
4. **Google Drive:** Same backup behavior as v2 when `BACKUP_TO_GOOGLE_DRIVE` is True on Colab.
- **Checkpoints / resume:** frequent `save_steps` on the quick path; default checkpoint folder moves to **Google Drive** on Colab when Drive is mounted so you can reconnect and **continue training** without restarting from scratch.



In [ ]:
# Validate OpenEnv installation (hackathon compliance)
try:
    import openenv
    print(f"✅ OpenEnv {openenv.__version__} installed")
except ImportError:
    print("⚠️  OpenEnv not yet installed (will be installed in next cell)")

print("✅ OpenEnv (latest release) check passed — hackathon compliance")

In [ ]:
!pip install -q openenv>=0.2.3
!pip install -q transformers==4.56.1 datasets==4.3.0
!pip install -q bitsandbytes hf_transfer tyro unsloth_zoo mergekit
!pip install -U unsloth[colab-new]
!pip install -q "trl>=0.18.2,<=0.24.0" pydantic requests matplotlib numpy pandas pyyaml accelerate

## Configuration & HF Space connectivity

The next code cell adds **multi-incident** defaults and **`run_id`** for scoped learning curves. Override with:

- `NEXUS_INCIDENT_POOL` — e.g. `INC003,INC008,INC004` (comma-separated). Ignored if `NEXUS_MULTI_INCIDENT=false`.
- `NEXUS_MULTI_INCIDENT` — `false` to train against **`NEXUS_INCIDENT_ID`** only (v2-style).
- `NEXUS_INCIDENT_ROTATION` — `round_robin` (default) or `random`.
- `NEXUS_GRPO_RUN_ID` — string passed to `POST /reset` as `run_id` (default `colab_grpo_enhanced`).

`REWARD_MAX_STEPS` default is **35** so mixed pools have headroom vs INC003-only (28).
**Checkpoints:** On Colab with Drive mounted, weights save under `.../NEXUS_GRPO_backups/active_grpo_checkpoints` by default so a disconnect does not wipe them. Re-run the training cell to **resume** from the latest step (`NEXUS_FORCE_FRESH=true` starts over). Set `NEXUS_GRPO_OUTPUT_DIR` to override the directory.

---

## Colab free tier — reducing disconnect / expiry pain

**Free Colab cannot be guaranteed** to stay alive for hours (idle limits, preemption, daily caps). Mitigations:

1. **Drive checkpoints + resume** (this notebook): mount Drive in the config cell; **`GRPO_OUTPUT_DIR`** defaults to **`.../NEXUS_GRPO_backups/active_grpo_checkpoints`** on Colab when Drive is present. After a disconnect, **re-run setup cells in order**, then training — **`trainer.train(resume_from_checkpoint=...)`** picks up the latest step unless **`NEXUS_FORCE_FRESH=true`**.
2. **Shorter runs:** **`ONE_ROUND_TRAINING = True`** or fewer prompts per session; continue later with resume instead of one very long run.
3. **Frequent `save_steps`** on the quick path so less work is lost.
4. **Stable browser session:** avoid laptop sleep; keep the Colab tab in a focused window on a reliable network while training runs.
5. **Colab Pro / Pro+** if you need longer single sessions.

**Disable HF checkpoints entirely:** set **`NEXUS_ENABLE_CHECKPOINTS=false`** (no checkpoint files, no resume; saves Drive space).



In [ ]:
import os
import requests


def _env(name, default=None, cast=str):
    raw = os.environ.get(name)
    if raw is None or raw == "":
        return default
    if cast is bool:
        return str(raw).lower() in ("1", "true", "yes", "y", "on")
    return cast(raw)


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


IN_COLAB = _in_colab()

# ---------------------------------------------------------------------------
# Notebook configuration — edit defaults here or set environment variables.
# Enhanced notebook extras:
#   NEXUS_INCIDENT_POOL     — comma-separated case ids (default: INC003,INC008,INC001)
#   NEXUS_MULTI_INCIDENT    — false → use only NEXUS_INCIDENT_ID (v2-style single task)
#   NEXUS_INCIDENT_ROTATION — round_robin | random
#   NEXUS_GRPO_RUN_ID       — POST /reset run_id for scoped /learning-curve and /metrics
#
# See grpo_colab_v2.ipynb for the full list of NEXUS_* vars (same as here).
# NEXUS_ENABLE_CHECKPOINTS (true/false) — false: no HF checkpoint files, no resume
# ---------------------------------------------------------------------------

BASE_URL = _env(
    "NEXUS_HF_SPACE_URL",
    "https://kunalkachru23-nexus-enhanced-stage.hf.space",
).rstrip("/")

TRAINING_INCIDENT_ID = _env("NEXUS_INCIDENT_ID", "INC003")
USE_MULTI_INCIDENT = _env("NEXUS_MULTI_INCIDENT", True, cast=bool)
_pool_raw = os.environ.get("NEXUS_INCIDENT_POOL")
if not USE_MULTI_INCIDENT:
    TRAINING_INCIDENT_POOL = [TRAINING_INCIDENT_ID]
elif _pool_raw and _pool_raw.strip():
    TRAINING_INCIDENT_POOL = [x.strip() for x in _pool_raw.split(",") if x.strip()]
else:
    TRAINING_INCIDENT_POOL = ["INC003", "INC008", "INC001"]

INCIDENT_ROTATION = _env("NEXUS_INCIDENT_ROTATION", "round_robin", cast=str).strip().lower()
if INCIDENT_ROTATION not in ("round_robin", "random"):
    INCIDENT_ROTATION = "round_robin"

GRPO_RUN_ID = _env("NEXUS_GRPO_RUN_ID", "colab_grpo_enhanced", cast=str).strip() or "colab_grpo_enhanced"

ONE_ROUND_TRAINING = _env("NEXUS_ONE_ROUND", True, cast=bool)

N_PROMPTS_QUICK = _env("NEXUS_QUICK_PROMPTS", 8, int)
N_PROMPTS_FULL = _env("NEXUS_FULL_PROMPTS", 60, int)

MODEL_NAME = _env("NEXUS_MODEL_NAME", "unsloth/Qwen2.5-1.5B-Instruct")
MAX_SEQ_LENGTH = _env("NEXUS_MAX_SEQ_LENGTH", 1024, int)

# Headroom for mixed incidents (INC007 etc. if you extend the pool).
REWARD_MAX_STEPS = _env("NEXUS_REWARD_MAX_STEPS", 35, int)

# Training checkpoints: NEXUS_RESUME=false disables resume; NEXUS_FORCE_FRESH=true ignores on-disk checkpoints.
CHECKPOINT_RESUME = _env("NEXUS_RESUME", True, cast=bool)
FORCE_FRESH_RUN = _env("NEXUS_FORCE_FRESH", False, cast=bool)
# Set NEXUS_ENABLE_CHECKPOINTS=false to skip saving HF checkpoints and never resume (saves disk / Drive).
ENABLE_CHECKPOINTS = _env("NEXUS_ENABLE_CHECKPOINTS", True, cast=bool)

# GRPO_OUTPUT_DIR is finalized after optional Google Drive mount (survives Colab disconnect when on Drive).
GRPO_LEARNING_RATE = _env("NEXUS_GRPO_LR", 5e-5, float)
GRPO_NUM_TRAIN_EPOCHS = _env("NEXUS_GRPO_EPOCHS", 1, int)
GRPO_LOGGING_STEPS = _env("NEXUS_GRPO_LOGGING_STEPS", 1, int)

GRPO_BATCH_QUICK = _env("NEXUS_GRPO_BATCH_QUICK", 1, int)
GRPO_BATCH_FULL = _env("NEXUS_GRPO_BATCH_FULL", 2, int)
GRPO_NUM_GEN_QUICK = _env("NEXUS_GRPO_NUM_GEN_QUICK", 2, int)
GRPO_NUM_GEN_FULL = _env("NEXUS_GRPO_NUM_GEN_FULL", 4, int)
GRPO_SAVE_STEPS_QUICK = _env("NEXUS_GRPO_SAVE_STEPS_QUICK", 5, int)  # frequent saves for short Colab runs
GRPO_SAVE_STEPS_FULL = _env("NEXUS_GRPO_SAVE_STEPS_FULL", 10, int)

PLOT_BASELINE_DEFAULT = _env("NEXUS_PLOT_BASELINE", 0.265, float)
PLOT_OUTPUT_DIR = _env("NEXUS_PLOT_DIR", "/content")

PEFT_R = _env("NEXUS_PEFT_R", 16, int)
PEFT_ALPHA = _env("NEXUS_PEFT_ALPHA", 16, int)
PEFT_DROPOUT = _env("NEXUS_PEFT_DROPOUT", 0.05, float)

BACKUP_TO_GOOGLE_DRIVE = _env("NEXUS_BACKUP_TO_DRIVE", IN_COLAB, cast=bool)
GOOGLE_DRIVE_BACKUP_ROOT = _env(
    "NEXUS_DRIVE_BACKUP_ROOT",
    "/content/drive/MyDrive/NEXUS_GRPO_backups",
).rstrip("/")

# Incident-agnostic IC prompts (pool rotates env episodes separately).
TRAINING_PROMPTS = [
    "You are an Incident Commander. Multiple services show elevated error rates and latency. What is your first assessment?",
    "Hypothesis: cascading failure from one upstream dependency. How do you triage and what do you delegate to L2?",
    "Customer impact is growing. What notifications or escalations do you order in the first 15 minutes?",
    "You need coalition alignment on root cause between ML, platform, and SRE. How do you frame the decision?",
    "Runbooks are available but incomplete telemetry. How do you sequence investigation vs mitigation?",
    "Executive EA scenario: conflicting calendar commitments and stakeholder pressure. How do you prioritize and communicate?",
    "Enterprise outage: revenue-impacting SLAs at risk. What is your mitigation ordering and rollback posture?",
    "Evidence suggests a config change rather than a code bug. What steps confirm or falsify that?",
    "Third-party vendor latency is spiking; internal retries mask errors. How do you uncover the true failure mode?",
    "After mitigation, what post-incident steps and confidence level do you report to stakeholders?",
]


def ensure_google_drive_mounted():
    """Mount Drive when backups are enabled (Colab only)."""
    if not BACKUP_TO_GOOGLE_DRIVE:
        return
    if not IN_COLAB:
        print("⚠️ BACKUP_TO_GOOGLE_DRIVE is True but not in Colab — skipping Drive mount.")
        return
    if os.path.isdir("/content/drive/MyDrive"):
        print("✅ Google Drive already mounted.")
        return
    from google.colab import drive
    print("📂 Mount Google Drive when prompted (artifacts copy to My Drive / NEXUS_GRPO_backups).")
    drive.mount("/content/drive")


def print_notebook_config():
    n = N_PROMPTS_QUICK if ONE_ROUND_TRAINING else N_PROMPTS_FULL
    mode = "one-round" if ONE_ROUND_TRAINING else "full"
    print("Notebook configuration (enhanced):")
    print(f"  BASE_URL = {BASE_URL}")
    print(f"  TRAINING_INCIDENT_POOL = {TRAINING_INCIDENT_POOL}")
    print(f"  USE_MULTI_INCIDENT = {USE_MULTI_INCIDENT}")
    print(f"  INCIDENT_ROTATION = {INCIDENT_ROTATION}")
    print(f"  GRPO_RUN_ID = {GRPO_RUN_ID}")
    print(f"  TRAINING_INCIDENT_ID (fallback / single-mode) = {TRAINING_INCIDENT_ID}")
    print(f"  ONE_ROUND_TRAINING = {ONE_ROUND_TRAINING}  ({mode}, target prompts={n})")
    print(f"  REWARD_MAX_STEPS = {REWARD_MAX_STEPS}")
    print(f"  MODEL_NAME = {MODEL_NAME}")
    print(f"  GRPO_OUTPUT_DIR = {GRPO_OUTPUT_DIR}")
    print(f"  CHECKPOINT_RESUME (NEXUS_RESUME) = {CHECKPOINT_RESUME}")
    print(f"  ENABLE_CHECKPOINTS (NEXUS_ENABLE_CHECKPOINTS) = {ENABLE_CHECKPOINTS}")
    print(f"  FORCE_FRESH_RUN (NEXUS_FORCE_FRESH) = {FORCE_FRESH_RUN}")
    print(f"  PLOT_OUTPUT_DIR = {PLOT_OUTPUT_DIR}")
    print(f"  IN_COLAB = {IN_COLAB}")
    print(f"  BACKUP_TO_GOOGLE_DRIVE = {BACKUP_TO_GOOGLE_DRIVE}")
    if BACKUP_TO_GOOGLE_DRIVE:
        print(f"  GOOGLE_DRIVE_BACKUP_ROOT = {GOOGLE_DRIVE_BACKUP_ROOT}")


ensure_google_drive_mounted()


def _finalize_checkpoint_dir():
    """Pick GRPO_OUTPUT_DIR: explicit env, else Drive-backed path on Colab, else local ./grpo_checkpoints."""
    global GRPO_OUTPUT_DIR
    explicit = os.environ.get("NEXUS_GRPO_OUTPUT_DIR")
    if explicit and str(explicit).strip():
        GRPO_OUTPUT_DIR = str(explicit).strip()
    elif IN_COLAB and os.path.isdir("/content/drive/MyDrive"):
        GRPO_OUTPUT_DIR = os.path.join(GOOGLE_DRIVE_BACKUP_ROOT, "active_grpo_checkpoints")
    else:
        GRPO_OUTPUT_DIR = "./grpo_checkpoints"
    os.makedirs(GRPO_OUTPUT_DIR, exist_ok=True)


_finalize_checkpoint_dir()

print_notebook_config()

try:
    resp = requests.get(f"{BASE_URL}/health", timeout=5)
    if resp.status_code == 200:
        print("✅ HF Space is reachable")
        print(f"Response: {resp.json()}")
    else:
        print(f"❌ HF Space returned status {resp.status_code}")
except Exception as e:
    print(f"❌ Error connecting to HF Space: {e}")
    print(f"URL: {BASE_URL}")
    print("Make sure HF Space is deployed and running")


In [ ]:
import os
import shutil
import json as _json_backup
from datetime import datetime, timezone

import requests


class NexusRemoteEnv:
    """
    Remote NEXUS environment wrapper (OpenEnv v0.2.3 compatible).

    Calls HF Space API endpoints that expose OpenEnv interface.
    Enhanced: optional run_id on reset; learning curve can be scoped by run_id.
    """

    def reset(self, incident_id=None, run_id=None):
        """POST /reset - start new episode (OpenEnv reset contract)"""
        if incident_id is None:
            incident_id = TRAINING_INCIDENT_ID
        body = {
            "incident_id": incident_id,
            "difficulty": None,
            "seed": None,
        }
        rid = run_id if run_id is not None else GRPO_RUN_ID
        if rid:
            body["run_id"] = rid
        resp = requests.post(
            f"{BASE_URL}/reset",
            json=body,
            timeout=10,
        )
        resp.raise_for_status()
        data = resp.json()
        return data["session_id"], data["observation"]

    def step(self, session_id, action):
        """POST /step - execute IC action (OpenEnv step contract)"""
        resp = requests.post(
            f"{BASE_URL}/step/{session_id}",
            json=action,
            timeout=10,
        )
        resp.raise_for_status()
        data = resp.json()
        return data["observation"], data["reward"], data["done"], data["info"]

    def get_learning_curve(self, run_id=None):
        """GET /learning-curve — optional run_id scopes metrics to this Colab run."""
        params = {}
        rid = run_id if run_id is not None else GRPO_RUN_ID
        if rid:
            params["run_id"] = rid
        resp = requests.get(
            f"{BASE_URL}/learning-curve",
            params=params if params else None,
            timeout=10,
        )
        resp.raise_for_status()
        return resp.json()


env = NexusRemoteEnv()
print("✅ Environment interface ready (enhanced: run_id + scoped learning curve)")

_NEXUS_DRIVE_RUN_DIR = None


def _nexus_google_drive_run_dir():
    global _NEXUS_DRIVE_RUN_DIR
    if _NEXUS_DRIVE_RUN_DIR is not None:
        return _NEXUS_DRIVE_RUN_DIR
    if not BACKUP_TO_GOOGLE_DRIVE or not IN_COLAB:
        return None
    if not os.path.isdir("/content/drive/MyDrive"):
        return None
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%SZ")
    mode = "quick" if ONE_ROUND_TRAINING else "full"
    _NEXUS_DRIVE_RUN_DIR = os.path.join(
        GOOGLE_DRIVE_BACKUP_ROOT,
        f"run_enhanced_{mode}_{stamp}",
    )
    os.makedirs(_NEXUS_DRIVE_RUN_DIR, exist_ok=True)
    return _NEXUS_DRIVE_RUN_DIR


def backup_nexus_artifacts_to_drive(reason="manual", *, include_learning_curve=True):
    """Copy GRPO checkpoints, PNG plots (if present), learning curve JSON, manifest → Google Drive."""
    if not BACKUP_TO_GOOGLE_DRIVE:
        print(f"[Drive backup:{reason}] skipped (BACKUP_TO_GOOGLE_DRIVE=False)")
        return None
    if not IN_COLAB:
        print(f"[Drive backup:{reason}] skipped (not Colab)")
        return None
    if not os.path.isdir("/content/drive/MyDrive"):
        print(f"[Drive backup:{reason}] skipped — mount Drive in the config cell first")
        return None
    dest = _nexus_google_drive_run_dir()
    if dest is None:
        print(f"[Drive backup:{reason}] skipped (could not create run folder)")
        return None

    if os.path.isdir(GRPO_OUTPUT_DIR):
        tgt = os.path.join(dest, "grpo_checkpoints")
        shutil.copytree(GRPO_OUTPUT_DIR, tgt, dirs_exist_ok=True)
        print(f"  ✅ checkpoints → {tgt}")

    for name in ("training_analysis.png", "reward_curves_hires.png"):
        src = os.path.join(PLOT_OUTPUT_DIR, name)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(dest, name))
            print(f"  ✅ plot {name}")

    if include_learning_curve:
        try:
            curve = env.get_learning_curve()
            with open(os.path.join(dest, "learning_curve.json"), "w") as f:
                _json_backup.dump(curve, f, indent=2)
            print("  ✅ learning_curve.json")
        except Exception as e:
            print(f"  ⚠️ learning curve fetch failed: {e}")

    manifest = {
        "reason": reason,
        "notebook": "grpo_colab_enhanced",
        "BASE_URL": BASE_URL,
        "GRPO_RUN_ID": GRPO_RUN_ID,
        "TRAINING_INCIDENT_POOL": TRAINING_INCIDENT_POOL,
        "INCIDENT_ROTATION": INCIDENT_ROTATION,
        "ONE_ROUND_TRAINING": ONE_ROUND_TRAINING,
        "MODEL_NAME": MODEL_NAME,
        "GRPO_OUTPUT_DIR": GRPO_OUTPUT_DIR,
        "PLOT_OUTPUT_DIR": PLOT_OUTPUT_DIR,
    }
    with open(os.path.join(dest, "run_manifest.json"), "w") as f:
        _json_backup.dump(manifest, f, indent=2)
    print(f"\n📦 Drive backup ({reason}): {dest}\n")
    return dest


In [ ]:
import random
import re

env = NexusRemoteEnv()

_reward_rr_counter = 0


def _pick_incident_for_episode():
    """Choose incident id for one GRPO reward episode."""
    global _reward_rr_counter
    pool = TRAINING_INCIDENT_POOL
    if not pool:
        return TRAINING_INCIDENT_ID
    if INCIDENT_ROTATION == "random":
        return random.choice(pool)
    choice = pool[_reward_rr_counter % len(pool)]
    _reward_rr_counter += 1
    return choice


def parse_ic_action(completion_text):
    """Parse IC action from LLM completion"""
    situation = completion_text[:200] if len(completion_text) > 200 else completion_text
    situation = situation.split("==")[0].strip() if "==" in situation else situation.strip()

    hypothesis_match = re.search(r"hypothesis[:\s]+([^\n.]+)", completion_text, re.IGNORECASE)
    hypothesis = hypothesis_match.group(1).strip() if hypothesis_match else situation[:50]

    coalition_match = re.search(r"coalition[:\s]+([^\n.]+)", completion_text, re.IGNORECASE)
    coalition_vote = coalition_match.group(1).strip() if coalition_match else None

    action = {
        "situation_assessment": situation,
        "hypothesis": hypothesis,
        "coalition_vote": coalition_vote,
        "l1_directive": {"action": "send_notification", "parameters": {}, "reasoning": ""},
        "l2_directive": {"action": "check_all_alerts", "parameters": {}, "reasoning": ""},
        "sre_directive": {"action": "list_runbooks", "parameters": {}, "reasoning": ""},
        "pm_directive": {"action": "check_sla_status", "parameters": {}, "reasoning": ""},
        "severity_assessment": "p2",
        "resolution_confidence": 0.75,
        "escalation_required": False,
    }

    return action


def reward_fn(completions, **kwargs):
    """Reward function: run episodes until completion; incident from pool per completion."""
    rewards = []

    for completion_text in completions:
        try:
            incident_id = _pick_incident_for_episode()
            session_id, obs = env.reset(incident_id=incident_id)
            done = False
            step_count = 0

            while not done and step_count < REWARD_MAX_STEPS:
                action = parse_ic_action(completion_text)
                obs, reward, done, info = env.step(session_id, action)
                step_count += 1

                if reward > 0:
                    break

            rewards.append(reward)

        except Exception:
            rewards.append(0.0)

    return rewards


print(
    "✅ Reward function defined (pool=",
    TRAINING_INCIDENT_POOL,
    ", rotation=",
    INCIDENT_ROTATION,
    ", run_id=",
    GRPO_RUN_ID,
    ")",
    sep="",
)


## Cell 5: Load Model and Configure GRPO

In [ ]:
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
import torch

_initial_batch = GRPO_BATCH_QUICK if ONE_ROUND_TRAINING else GRPO_BATCH_FULL
_initial_gen = GRPO_NUM_GEN_QUICK if ONE_ROUND_TRAINING else GRPO_NUM_GEN_FULL

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=PEFT_R,
    lora_alpha=PEFT_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=PEFT_DROPOUT,
    bias="none",
    use_gradient_checkpointing=True,
    use_rslora=True,
)

config = GRPOConfig(
    output_dir=GRPO_OUTPUT_DIR,
    learning_rate=GRPO_LEARNING_RATE,
    per_device_train_batch_size=_initial_batch,
    num_train_epochs=GRPO_NUM_TRAIN_EPOCHS,
    num_generations=_initial_gen,
    logging_steps=GRPO_LOGGING_STEPS,
    save_steps=GRPO_SAVE_STEPS_QUICK if ONE_ROUND_TRAINING else GRPO_SAVE_STEPS_FULL,
)

print("✅ Model loaded and GRPO configured")


## MAIN GRPO training

Training scope is controlled in the **configuration cell** (same notebook, earlier):

- Set **`ONE_ROUND_TRAINING = True`** there (or `os.environ["NEXUS_ONE_ROUND"]="true"`) for the quick path (**`N_PROMPTS_QUICK`** prompts, lighter batch/generations).
- Set **`ONE_ROUND_TRAINING = False`** (or `NEXUS_ONE_ROUND=false`) for the full run (**`N_PROMPTS_FULL`** prompts).

**Enhanced notebook:** incident pool and `GRPO_RUN_ID` are also configured there. Plots fetch **`/learning-curve?run_id=...`** so curves match this run when the Space supports scoped metrics.


In [ ]:
from datasets import Dataset
import os
from transformers.trainer_utils import get_last_checkpoint

n_target = N_PROMPTS_QUICK if ONE_ROUND_TRAINING else N_PROMPTS_FULL

print("\n" + "=" * 70)
if ONE_ROUND_TRAINING:
    print(f"🚀 GRPO — ONE ROUND ({n_target} prompts, fast path)")
else:
    print(f"🚀 GRPO — FULL RUN ({n_target} prompts)")
print("=" * 70)
print("\nConfiguration:")
print(f"  • Model: {MODEL_NAME}")
print(f"  • Dataset rows: {n_target}")
print(f"  • Environment: {BASE_URL}")
print(f"  • Incident pool: {TRAINING_INCIDENT_POOL} ({INCIDENT_ROTATION})")
print(f"  • GRPO_RUN_ID (metrics scope): {GRPO_RUN_ID}")
print(f"  • Checkpoints dir: {GRPO_OUTPUT_DIR}")
print("  • Adjust settings in the configuration cell (or NEXUS_* env vars).")
print("\nMonitor dashboard:")
print(f"  {BASE_URL}/")
print("=" * 70 + "\n")

pool = TRAINING_PROMPTS
repeats = (n_target + len(pool) - 1) // len(pool)
prompts = (pool * repeats)[:n_target]

train_dataset = Dataset.from_dict({"prompt": prompts})

config.num_train_epochs = GRPO_NUM_TRAIN_EPOCHS
config.per_device_train_batch_size = GRPO_BATCH_QUICK if ONE_ROUND_TRAINING else GRPO_BATCH_FULL
config.num_generations = GRPO_NUM_GEN_QUICK if ONE_ROUND_TRAINING else GRPO_NUM_GEN_FULL
config.logging_steps = GRPO_LOGGING_STEPS
config.save_steps = GRPO_SAVE_STEPS_QUICK if ONE_ROUND_TRAINING else GRPO_SAVE_STEPS_FULL
if not ENABLE_CHECKPOINTS:
    config.save_strategy = "no"

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=config,
    train_dataset=train_dataset,
)

resume_ckpt = None
if ENABLE_CHECKPOINTS and CHECKPOINT_RESUME and not FORCE_FRESH_RUN:
    resume_ckpt = get_last_checkpoint(GRPO_OUTPUT_DIR)
if resume_ckpt:
    print(f"📂 Resuming training from: {resume_ckpt}")
else:
    print("📂 Starting training fresh (no checkpoint, or NEXUS_RESUME=false, or NEXUS_FORCE_FRESH=true)")

print(f"📊 Dataset: {len(train_dataset)} prompts")
print("⏳ Training started...")
trainer.train(resume_from_checkpoint=resume_ckpt)

print("\n" + "=" * 70)
print("✅ Training step finished")
print("=" * 70)
print(f"Dashboard: {BASE_URL}/")
print(f"Learning curve API: {BASE_URL}/learning-curve")
print("▶️ Run next cell to plot results.")
print("=" * 70)

backup_nexus_artifacts_to_drive("post_training", include_learning_curve=True)


In [ ]:
import os
import matplotlib.pyplot as plt

os.makedirs(PLOT_OUTPUT_DIR, exist_ok=True)

print("\n" + "=" * 70)
print("📊 FETCHING REAL TRAINING DATA FROM HF SPACE")
print("=" * 70)
print(f"Using run_id filter: {GRPO_RUN_ID}")

env = NexusRemoteEnv()
learning_data = env.get_learning_curve()  # scoped by GRPO_RUN_ID on server

rewards = learning_data.get("rewards", [])

if rewards:
    episodes = list(range(1, len(rewards) + 1))
    baseline = learning_data.get("baseline", PLOT_BASELINE_DEFAULT)

    rolling_avg = []
    for i in range(len(rewards)):
        start = max(0, i - 4)
        chunk = rewards[start : i + 1]
        rolling_avg.append(sum(chunk) / len(chunk))

    fig = plt.figure(figsize=(16, 10))

    ax1 = plt.subplot(2, 2, 1)
    ax1.plot(episodes, rewards, "o-", label="Episode Reward", color="#0ea5e9", markersize=6, linewidth=2, alpha=0.7)
    ax1.plot(episodes, rolling_avg, "-", label="5-Episode Rolling Avg", color="#10b981", linewidth=3)
    ax1.axhline(y=baseline, color="#ef4444", linestyle="--", linewidth=2, label=f"Baseline: {baseline:.3f}")
    ax1.set_xlabel("Episode", fontsize=11)
    ax1.set_ylabel("Reward Score", fontsize=11)
    ax1.set_title("Training Progress: Reward Curves", fontsize=12, fontweight="bold")
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1)

    improvement_pct = [((r - baseline) / baseline * 100) for r in rewards]
    ax2 = plt.subplot(2, 2, 2)
    ax2.bar(episodes, improvement_pct, color="#06b6d4", alpha=0.7, edgecolor="#0ea5e9")
    ax2.axhline(y=0, color="black", linestyle="-", linewidth=1)
    ax2.set_xlabel("Episode", fontsize=11)
    ax2.set_ylabel("Improvement (%)", fontsize=11)
    ax2.set_title("Improvement vs Baseline", fontsize=12, fontweight="bold")
    ax2.grid(True, alpha=0.3, axis="y")

    cumulative_avg = []
    for i in range(len(rewards)):
        cumulative_avg.append(sum(rewards[: i + 1]) / (i + 1))

    ax3 = plt.subplot(2, 2, 3)
    ax3.plot(episodes, cumulative_avg, "o-", label="Cumulative Average", color="#8b5cf6", linewidth=2.5)
    ax3.axhline(y=baseline, color="#ef4444", linestyle="--", linewidth=2, label=f"Baseline: {baseline:.3f}")
    ax3.set_xlabel("Episode", fontsize=11)
    ax3.set_ylabel("Average Reward", fontsize=11)
    ax3.set_title("Cumulative Learning Progress", fontsize=12, fontweight="bold")
    ax3.legend(fontsize=10)
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim(0, 1)

    ax4 = plt.subplot(2, 2, 4)
    ax4.axis("off")

    best_reward = max(rewards)
    avg_reward = sum(rewards) / len(rewards)
    improvement_from_baseline = ((avg_reward - baseline) / baseline * 100)
    last_5_avg = sum(rewards[-5:]) / min(5, len(rewards))

    summary_text = f"""
TRAINING SUMMARY

📊 Episodes: {len(rewards)}
🔵 Baseline: {baseline:.4f}
📈 Average: {avg_reward:.4f}
⭐ Best: {best_reward:.4f}
📉 Worst: {min(rewards):.4f}

📊 Improvement: +{improvement_from_baseline:.1f}%
📌 Last 5 Avg: {last_5_avg:.4f}
    """

    ax4.text(
        0.1,
        0.9,
        summary_text,
        transform=ax4.transAxes,
        fontsize=11,
        verticalalignment="top",
        fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="#1e293b", alpha=0.8, edgecolor="#0ea5e9"),
    )

    plt.suptitle("NEXUS Enhanced — Complete Training Analysis", fontsize=14, fontweight="bold", y=0.995)
    plt.tight_layout()

    print("\n📁 Saving visualizations...")
    p1 = os.path.join(PLOT_OUTPUT_DIR, "training_analysis.png")
    p2 = os.path.join(PLOT_OUTPUT_DIR, "reward_curves_hires.png")
    plt.savefig(p1, dpi=150, bbox_inches="tight")
    print(f"  ✅ {p1} (4-panel comprehensive view)")

    fig_single, ax = plt.subplots(figsize=(14, 7))
    ax.plot(episodes, rewards, "o-", label="Episode Reward", color="#0ea5e9", markersize=7, linewidth=2.5, alpha=0.8)
    ax.plot(episodes, rolling_avg, "-", label="5-Episode Rolling Avg", color="#10b981", linewidth=3.5)
    ax.fill_between(episodes, rewards, alpha=0.15, color="#0ea5e9")
    ax.axhline(y=baseline, color="#ef4444", linestyle="--", linewidth=2.5, label=f"Baseline: {baseline:.3f}")
    ax.set_xlabel("Episode", fontsize=12, fontweight="bold")
    ax.set_ylabel("Reward Score", fontsize=12, fontweight="bold")
    ax.set_title("NEXUS Enhanced GRPO Training — Reward Progression", fontsize=13, fontweight="bold")
    ax.legend(fontsize=11, loc="lower right")
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.set_ylim(-0.05, 1.05)
    plt.tight_layout()
    plt.savefig(p2, dpi=200, bbox_inches="tight")
    print(f"  ✅ {p2} (high-res)")

    plt.show()

    print("\n" + "=" * 70)
    print("📈 FINAL TRAINING RESULTS")
    print("=" * 70)
    print(f"\n{'Metric':<35} {'Value':<20}")
    print("-" * 55)
    print(f"{'Total Episodes':<35} {len(rewards):<20}")
    print(f"{'Baseline (Untrained)':<35} {baseline:.4f}{'':>10}")
    print(f"{'Average Reward':<35} {avg_reward:.4f}{'':>10}")
    print(f"{'Best Reward':<35} {best_reward:.4f}{'':>10}")
    print(f"{'Worst Reward':<35} {min(rewards):.4f}{'':>10}")
    print(f"{'Overall Improvement':<35} +{improvement_from_baseline:.1f}%{'':>10}")
    print(f"{'Last 5 Episodes Average':<35} {last_5_avg:.4f}{'':>10}")
    print("-" * 55)

    if len(rewards) >= 10:
        early_avg = sum(rewards[:5]) / 5
        late_avg = sum(rewards[-5:]) / 5
        print(f"\n{'Early Phase (Ep 1-5) Avg':<35} {early_avg:.4f}")
        print(f"{'Late Phase (Ep -5) Avg':<35} {late_avg:.4f}")
        learning_status = "✅ Learning" if late_avg > early_avg else "⚠️  Plateau"
        print(f"{'Status':<35} {learning_status:<20}")

    print("\n" + "=" * 70)
    print("✅ COMPLETE!")
    print("=" * 70)

    backup_nexus_artifacts_to_drive("post_plots", include_learning_curve=True)

else:
    print("\n❌ No episode data found")
    print("⏳ Training may still be running...")
    print("💡 Rerun this cell in a few minutes")
    print(f"📊 Live: {BASE_URL}/learning-curve")
    backup_nexus_artifacts_to_drive("no_rewards_yet", include_learning_curve=True)
